In [1]:
import pandas as pd

df = pd.read_csv("../data/wilde_fire_data_county.csv")

In [2]:
df[df["incident_date_created"].between("2015-01-01", "2025-12-31")].value_counts(
    "incident_county"
).head(10)

incident_county
Riverside          320
Fresno             171
San Diego          165
Kern               165
San Luis Obispo    117
Los Angeles        113
San Bernardino     112
Siskiyou           104
Butte               97
Shasta              92
Name: count, dtype: int64

In [3]:
len(df[df["incident_date_created"].between("2015-01-01", "2025-12-31")])

3147

In [4]:
import numpy as np
import pandas as pd
import folium
from folium.plugins import MarkerCluster

df["incident_date_created"] = pd.to_datetime(
    df["incident_date_created"], errors="coerce"
)

ca_incidents = df[
    ~((df["incident_longitude"] == 1.0) & (df["incident_latitude"] == 1.0))
    & (df["incident_longitude"].between(-125.0, -114.0))
    & (df["incident_latitude"].between(32.0, 42.0))
    & (df["incident_date_created"].between("2025-01-01", "2025-12-31"))
].copy()

ca_incidents["log_acres"] = np.log1p(ca_incidents["incident_acres_burned"])

m = folium.Map(location=[37.2, -119.5], zoom_start=6, tiles="CartoDB positron")
marker_cluster = MarkerCluster().add_to(m)

for _, row in ca_incidents.iterrows():
    popup_text = f"""
    <b>Incident:</b> {row.get("incident_name", "Unknown")}<br>
    <b>Date:</b> {row["incident_date_created"].date() if pd.notnull(row["incident_date_created"]) else "Unknown"}<br>
    <b>Acres Burned:</b> {row.get("incident_acres_burned", "Unknown")}
    """

    folium.CircleMarker(
        location=[row["incident_latitude"], row["incident_longitude"]],
        radius=max(3, row["log_acres"] * 2),
        popup=popup_text,
        color="red",
        fill=True,
        fill_color="red",
        fill_opacity=0.6,
    ).add_to(marker_cluster)

m.save("../results/california_wildfire_map_2025.html")
m


In [5]:
import numpy as np
import pandas as pd
import folium
from branca.colormap import linear

df["incident_date_created"] = pd.to_datetime(
    df["incident_date_created"], errors="coerce"
)

ca_incidents = df[
    ~((df["incident_longitude"] == 1.0) & (df["incident_latitude"] == 1.0))
    & (df["incident_longitude"].between(-125.0, -114.0))
    & (df["incident_latitude"].between(32.0, 42.0))
    & (df["incident_date_created"].between("2018-01-01", "2018-12-31"))
].copy()

print(f"Found {len(ca_incidents)} incidents in California")

# safer numeric conversion
ca_incidents["incident_acres_burned"] = pd.to_numeric(
    ca_incidents["incident_acres_burned"], errors="coerce"
)
ca_incidents = ca_incidents.dropna(
    subset=["incident_latitude", "incident_longitude", "incident_acres_burned"]
)

# log scale for nicer bubble sizing
ca_incidents["log_acres"] = np.log1p(ca_incidents["incident_acres_burned"])

# base map
m = folium.Map(location=[37.2, -119.5], zoom_start=6, tiles="CartoDB positron")

# color scale
colormap = linear.YlOrRd_09.scale(
    ca_incidents["incident_acres_burned"].min(),
    ca_incidents["incident_acres_burned"].max(),
)
colormap.caption = "Acres Burned"

for _, row in ca_incidents.iterrows():
    incident_name = row.get("incident_name", "Unknown")
    county = row.get("incident_county", "Unknown")
    acres = row["incident_acres_burned"]
    date = (
        row["incident_date_created"].strftime("%Y-%m-%d")
        if pd.notnull(row["incident_date_created"])
        else "Unknown"
    )

    popup_html = f"""
    <b>Incident:</b> {incident_name}<br>
    <b>Date:</b> {date}<br>
    <b>County:</b> {county}<br>
    <b>Acres Burned:</b> {acres:,.0f}
    """

    folium.CircleMarker(
        location=[row["incident_latitude"], row["incident_longitude"]],
        radius=max(3, row["log_acres"] * 1.8),
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=f"{incident_name}: {acres:,.0f} acres",
        color=colormap(acres),
        fill=True,
        fill_color=colormap(acres),
        fill_opacity=0.7,
        weight=1,
    ).add_to(m)

colormap.add_to(m)
m.save("../results/california_wildfire_map_2025_bubble.html")
m

Found 302 incidents in California


In [6]:
import pandas as pd
import plotly.figure_factory as ff


df["incident_date_created"] = pd.to_datetime(
    df["incident_date_created"], errors="coerce"
)

ca_incidents = df[
    ~((df["incident_longitude"] == 1.0) & (df["incident_latitude"] == 1.0))
    & df["incident_longitude"].between(-125.0, -114.0)
    & df["incident_latitude"].between(32.0, 42.0)
    & df["incident_date_created"].between("2015-01-01", "2025-12-31")
].copy()

ca_incidents["incident_county"] = (
    ca_incidents["incident_county"]
    .astype(str)
    .str.strip()
    .str.title()
    .replace({"Nan": "Unknown"})
)

county_counts = (
    ca_incidents[ca_incidents["incident_county"] != "Unknown"]
    .groupby("incident_county")
    .size()
    .reset_index(name="incident_count")
)

ca_fips = {
    "Alameda": "06001",
    "Alpine": "06003",
    "Amador": "06005",
    "Butte": "06007",
    "Calaveras": "06009",
    "Colusa": "06011",
    "Contra Costa": "06013",
    "Del Norte": "06015",
    "El Dorado": "06017",
    "Fresno": "06019",
    "Glenn": "06021",
    "Humboldt": "06023",
    "Imperial": "06025",
    "Inyo": "06027",
    "Kern": "06029",
    "Kings": "06031",
    "Lake": "06033",
    "Lassen": "06035",
    "Los Angeles": "06037",
    "Madera": "06039",
    "Marin": "06041",
    "Mariposa": "06043",
    "Mendocino": "06045",
    "Merced": "06047",
    "Modoc": "06049",
    "Mono": "06051",
    "Monterey": "06053",
    "Napa": "06055",
    "Nevada": "06057",
    "Orange": "06059",
    "Placer": "06061",
    "Plumas": "06063",
    "Riverside": "06065",
    "Sacramento": "06067",
    "San Benito": "06069",
    "San Bernardino": "06071",
    "San Diego": "06073",
    "San Francisco": "06075",
    "San Joaquin": "06077",
    "San Luis Obispo": "06079",
    "San Mateo": "06081",
    "Santa Barbara": "06083",
    "Santa Clara": "06085",
    "Santa Cruz": "06087",
    "Shasta": "06089",
    "Sierra": "06091",
    "Siskiyou": "06093",
    "Solano": "06095",
    "Sonoma": "06097",
    "Stanislaus": "06099",
    "Sutter": "06101",
    "Tehama": "06103",
    "Trinity": "06105",
    "Tulare": "06107",
    "Tuolumne": "06109",
    "Ventura": "06111",
    "Yolo": "06113",
    "Yuba": "06115",
}

county_counts["fips"] = county_counts["incident_county"].map(ca_fips)
county_counts = county_counts.dropna(subset=["fips"])

colorscale = [
    "rgb(255,245,240)",
    "rgb(254,224,210)",
    "rgb(252,187,161)",
    "rgb(252,146,114)",
    "rgb(251,106,74)",
    "rgb(203,24,29)",
]

fig = ff.create_choropleth(
    fips=county_counts["fips"].tolist(),
    values=county_counts["incident_count"].tolist(),
    scope=["CA"],
    binning_endpoints=[5, 10, 20, 40, 80],
    colorscale=colorscale,
    county_outline={"color": "white", "width": 1},
    round_legend_values=True,
    legend_title="Wildfire Incidents",
    title="California Wildfire Incidents by County (2015-2025)",
)

fig.update_layout(template=None)
fig.write_html("../results/california_wildfire_choropleth_2015-2025.html")
fig.show()

/opt/anaconda3/envs/dsci510/lib/python3.13/site-packages/plotly/figure_factory/_county_choropleth.py:808: ShapelyDeprecationWarning: The 'type' attribute is deprecated, and will be removed in the future. You can use the 'geom_type' attribute instead.
  fips_polygon_map[f].type
/opt/anaconda3/envs/dsci510/lib/python3.13/site-packages/plotly/figure_factory/_county_choropleth.py:330: ShapelyDeprecationWarning: The 'type' attribute is deprecated, and will be removed in the future. You can use the 'geom_type' attribute instead.
  if fips_polygon_map[f].type == "Polygon":
/opt/anaconda3/envs/dsci510/lib/python3.13/site-packages/plotly/figure_factory/_county_choropleth.py:808: ShapelyDeprecationWarning: The 'type' attribute is deprecated, and will be removed in the future. You can use the 'geom_type' attribute instead.
  fips_polygon_map[f].type
/opt/anaconda3/envs/dsci510/lib/python3.13/site-packages/plotly/figure_factory/_county_choropleth.py:330: ShapelyDeprecationWarning: The 'type' attri

In [7]:
import pandas as pd
import plotly.express as px

# ── Wildfire Data ──────────────────────────────────────────────────────────────
df = pd.read_csv("../data/wilde_fire_data_county.csv")
top_counties = (
    df[df["incident_date_created"].between("2015-01-01", "2025-12-31")]
    .value_counts("incident_county")
    .head(10)
    .index.tolist()
)

# ── GDP Data ───────────────────────────────────────────────────────────────────
gdp_df = pd.read_csv("../data/CAGDP2_CA_2001_2024.csv")

year_cols = [col for col in gdp_df.columns if col.isdigit()]
ca_gdp = (
    gdp_df[gdp_df["Description"].str.strip() == "All industry total"]
    .assign(
        fips=lambda d: d["GeoFIPS"]
        .astype(str)
        .str.replace('"', "", regex=False)
        .str.strip(),
        county_name=lambda d: d["GeoName"]
        .astype(str)
        .str.replace(", CA", "", regex=False)
        .str.strip(),
    )
    .pipe(
        lambda d: d[
            d["fips"].str.match(r"^060\d{2}$", na=False) & (d["fips"] != "06000")
        ]
    )
)

ca_gdp[year_cols] = ca_gdp[year_cols].apply(pd.to_numeric, errors="coerce")

# Reshape wide → long, filter to top 10 wildfire counties
ts_df = (
    ca_gdp.melt(
        id_vars=["county_name", "fips"],
        value_vars=year_cols,
        var_name="year",
        value_name="gdp",
    )
    .assign(year=lambda d: d["year"].astype(int))
    .dropna(subset=["gdp"])
    .pipe(lambda d: d[d["county_name"].isin(top_counties)])
)

# ── Plot ───────────────────────────────────────────────────────────────────────
fig = px.line(
    ts_df,
    x="year",
    y="gdp",
    color="county_name",
    hover_name="county_name",
    title="All Industry GDP Over Time — Top 10 Wildfire Counties",
    labels={"year": "Year", "gdp": "GDP", "county_name": "County"},
)
fig.update_layout(xaxis=dict(tickmode="linear"), hovermode="x unified")
fig.show()

In [9]:
import pandas as pd
import plotly.express as px

# ── Wildfire Data ──────────────────────────────────────────────────────────────
df = pd.read_csv("../data/wilde_fire_data_county.csv")
top_counties = (
    df[
        df["incident_date_created"].between("2015-01-01", "2025-12-31")
        & (df["incident_county"].isin(["Shasta", "Butte"]))
    ]
    .value_counts("incident_county")
    .index.tolist()
)

# ── GDP Data ───────────────────────────────────────────────────────────────────
gdp_df = pd.read_csv("../data/CAGDP2_CA_2001_2024.csv")

year_cols = [col for col in gdp_df.columns if col.isdigit()]
ca_gdp = (
    gdp_df[gdp_df["Description"].str.strip() == "All industry total"]
    .assign(
        fips=lambda d: d["GeoFIPS"]
        .astype(str)
        .str.replace('"', "", regex=False)
        .str.strip(),
        county_name=lambda d: d["GeoName"]
        .astype(str)
        .str.replace(", CA", "", regex=False)
        .str.strip(),
    )
    .pipe(
        lambda d: d[
            d["fips"].str.match(r"^060\d{2}$", na=False) & (d["fips"] != "06000")
        ]
    )
)

ca_gdp[year_cols] = ca_gdp[year_cols].apply(pd.to_numeric, errors="coerce")

# Reshape wide → long, filter to Shasta and Butte
ts_df = (
    ca_gdp.melt(
        id_vars=["county_name", "fips"],
        value_vars=year_cols,
        var_name="year",
        value_name="gdp",
    )
    .assign(year=lambda d: d["year"].astype(int))
    .dropna(subset=["gdp"])
    .pipe(lambda d: d[d["county_name"].isin(["Shasta", "Butte"])])
)

fig = px.line(
    ts_df,
    x="year",
    y="gdp",
    color="county_name",
    hover_name="county_name",
    title="All Industry GDP Over Time — Shasta and Butte Counties",
    labels={"year": "Year", "gdp": "GDP", "county_name": "County"},
)
fig.update_layout(xaxis=dict(tickmode="linear"), hovermode="x unified")
fig.show()
fig.write_image("../results/All_Industry_GDP_Plot_Butte_Shasta.png")